# Heart disease model experiment

This notebook trains the heart disease model, compares it with other models, and exports it for
the server.

The server makes predictions on encrypted data using Paillier encryption. Paillier can add
encrypted numbers and multiply them by plain numbers, but it can't compare them. So the model has
to be a weighted sum of its inputs, which is what logistic regression is. Tree models and KNN
can't run this way.

## Loading the data

In the CSV, `target = 1` means **no** heart disease (we checked this against the original UCI
file). We flip it so that `1` means heart disease.

In [1]:
import json

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score, cross_validate, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [21]:
df = pd.read_csv("data/heart-disease.csv")
df.head(4)

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1


In [22]:
X = df.drop("target", axis=1)
y = (1 - df["target"]).rename("disease")

X.shape, y.value_counts().to_dict()

((303, 13), {0: 165, 1: 138})

In [18]:
pd.DataFrame(
    data=(1-df["target"]).rename("disease")
)

,disease
0,0
1,0
2,0
3,0
4,0
...,...
298,1
299,1
300,1
301,1


## Preparing the features

Five fields are categories, not amounts. For example, `cp = 2` is not "twice" `cp = 1`. We one-hot
encode them, which turns 13 fields into 27 inputs. The other fields are scaled to a similar
range.

In [24]:
CATEGORICAL = ["cp", "restecg", "slope", "ca", "thal"]
NUMERIC = [c for c in X.columns if c not in CATEGORICAL]

def build_pipeline(C=0.3):
    features = ColumnTransformer([
        ("num", StandardScaler(), NUMERIC),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
    ])
    return make_pipeline(features, LogisticRegression(C=C, max_iter=5000))

build_pipeline()

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'sex', 'trestbps',
                                                   'chol', 'fbs', 'thalach',
                                                   'exang', 'oldpeak']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['cp', 'restecg', 'slope',
                                                   'ca', 'thal'])])),
                ('logisticregression',
                 LogisticRegression(C=0.3, max_iter=5000))])

## Comparing models

Each model is trained and tested 50 times: 5-fold cross-validation, repeated 10 times with a fixed
seed. One test split isn't enough with only 303 patients.

Precision and recall are about finding patients with heart disease.

In [26]:
rf = RandomForestClassifier(n_estimators=205, random_state=0)

candidates = [
    ("Random Forest", clone(rf), "no"),
    ("KNN (k=5)", make_pipeline(StandardScaler(), KNeighborsClassifier()), "no"),
    ("Logistic regression, 13 inputs", make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)), "yes"),
    ("Logistic regression, 27 inputs", build_pipeline(), "yes"),
]

metrics = ["accuracy", "precision", "recall", "f1", "roc_auc"]
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)

rows, accuracy = [], {}

for name, model, works_encrypted in candidates:
    scores = cross_validate(model, X, y, cv=cv, scoring=metrics)
    accuracy[name] = scores["test_accuracy"]
    rows.append({"model": name, "works encrypted": works_encrypted,
                 **{m: f"{scores[f'test_{m}'].mean():.3f} +/- {scores[f'test_{m}'].std():.3f}" for m in metrics}})

pd.DataFrame(rows).set_index("model")

,works encrypted,accuracy,precision,recall,f1,roc_auc
model,,,,,,
Random Forest,no,0.822 +/- 0.046,0.827 +/- 0.064,0.777 +/- 0.084,0.798 +/- 0.056,0.904 +/- 0.034
KNN (k=5),no,0.816 +/- 0.049,0.833 +/- 0.068,0.753 +/- 0.093,0.786 +/- 0.064,0.877 +/- 0.043
"Logistic regression, 13 inputs",yes,0.830 +/- 0.040,0.860 +/- 0.066,0.755 +/- 0.077,0.800 +/- 0.049,0.897 +/- 0.039
"Logistic regression, 27 inputs",yes,0.851 +/- 0.050,0.859 +/- 0.059,0.809 +/- 0.096,0.830 +/- 0.062,0.918 +/- 0.035


Both models were tested on the same 50 splits, so we can compare them split by split.

In [27]:
diff = accuracy["Logistic regression, 27 inputs"] - accuracy["Random Forest"]
print(f"Average accuracy gain over the Random Forest: {diff.mean():+.3f}")
print(f"Splits won / tied / lost: {(diff > 0).sum()} / {(diff == 0).sum()} / {(diff < 0).sum()}")

Average accuracy gain over the Random Forest: +0.029
Splits won / tied / lost: 33 / 10 / 7


## Choosing C

`C` controls how strongly the weights are kept small. Lower means stronger.

In [30]:
pd.DataFrame(
    [{"C": C, "accuracy": cross_val_score(build_pipeline(C), X, y, cv=cv).mean().round(4)}
     for C in [0.03, 0.1, 0.3, 1, 3]]
).set_index("C")

,accuracy
C,
0.03,0.8297
0.10,0.8446
0.30,0.8508
1.00,0.8462
3.00,0.8422


## Why one test split isn't enough

The same Random Forest, tested on 30 different random 80/20 splits:

In [31]:
single = []
for seed in range(30):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed)
    single.append(clone(rf).fit(X_tr, y_tr).score(X_te, y_te))
single = np.array(single)
print(f"min {single.min():.3f}, median {np.median(single):.3f}, max {single.max():.3f}")

min 0.721, median 0.803, max 0.902


## Training the final model

We train on all 303 patients with `C = 0.3`.

In [32]:
pipeline = build_pipeline(C=0.3).fit(X, y)
cv_accuracy = cross_val_score(build_pipeline(C=0.3), X, y, cv=cv).mean()
print(f"Cross-validated accuracy: {cv_accuracy:.4f}")

Cross-validated accuracy: 0.8508


## Exporting the weights

The server only needs a list of weights and a constant. The scaling step is folded into the
weights, so the app can encrypt the raw measurements:

`w * (x - mean) / std` is the same as `(w / std) * x - w * mean / std`

The `features` list tells the app which 27 inputs to send and in what order.

In [33]:
scaler = pipeline[0].named_transformers_["num"]
encoder = pipeline[0].named_transformers_["cat"]
coef = pipeline[-1].coef_[0]

features, weights = [], []
intercept = float(pipeline[-1].intercept_[0])

for name, w, mean, std in zip(NUMERIC, coef, scaler.mean_, scaler.scale_):
    features.append({"column": name})
    weights.append(float(w / std))
    intercept -= float(w * mean / std)

i = len(NUMERIC)
for name, categories in zip(CATEGORICAL, encoder.categories_):
    for value in categories:
        features.append({"column": name, "equals": int(value)})
        weights.append(float(coef[i]))
        i += 1

def encode(row):
    """Turn one patient into the 27 inputs, the same way the app does."""
    return np.array([row[f["column"]] if "equals" not in f else float(row[f["column"]] == f["equals"])
                     for f in features])

# The exported weights must give exactly the same scores as the trained model.
scores = np.array([encode(row) for _, row in X.iterrows()]) @ np.array(weights) + intercept
assert np.allclose(scores, pipeline.decision_function(X))

model = {"model": "logistic_regression", "features": features, "weights": weights,
         "intercept": intercept, "cv_accuracy": round(float(cv_accuracy), 4)}
with open("../he_services/ml_models/heart_disease_logistic_regression.json", "w") as f:
    json.dump(model, f, indent=2)
print(f"Saved {len(weights)} weights")

Saved 27 weights


## Encrypted vs normal predictions

Every patient is encrypted with a 2048-bit key, sent to the Flask endpoint, and the result is
decrypted. The request contains only the public key and the encrypted values.

This takes a few minutes with `gmpy2` installed, and about an hour without it.

In [ ]:
import sys, time
import phe as paillier

sys.path.insert(0, "../he_services")
from app import app

client = app.test_client()
url = "/api/operations/ml/classification"
public_key, private_key = paillier.generate_paillier_keypair(n_length=2048)

start = time.perf_counter()
errors, same = [], 0

for _, row in X.iterrows():
    x = encode(row)
    body = {"public_key": {"n": str(public_key.n)},
            "values": [(str(c.ciphertext()), c.exponent) for c in map(public_key.encrypt, x.tolist())]}
    c, exponent = client.post(url, json=body).get_json()["values"][0]
    score = private_key.decrypt(paillier.EncryptedNumber(public_key, int(c), int(exponent)))
    expected = float(x @ np.array(weights) + intercept)
    errors.append(abs(score - expected))
    same += (score > 0) == (expected > 0)

print(f"{same}/{len(X)} encrypted predictions match")
print(f"Largest score difference: {max(errors):.2e}")
print(f"Time: {time.perf_counter() - start:.0f}s")

303/303 encrypted predictions match
Largest score difference: 1.78e-15
Time: 103s


## Result

The 27-input logistic regression is the best of the four: 85.1% accuracy, 0.809 recall and 0.918
ROC AUC. It beat the Random Forest on 33 of the 50 splits and lost on 7. One-hot encoding adds
about 2 points of accuracy over using raw codes. `C = 0.3` works best, but 0.1 to 1 are all close.

Encryption doesn't change any prediction.

Recall of 0.81 means it still misses about 1 in 5 patients with heart disease. The ± values show
how much results varied between splits. With 303 patients from one hospital, these numbers may not
hold for other groups.